# 🧠 DermaAI Vision - Entrenamiento del Modelo CNN

Este notebook entrena un modelo de clasificación de enfermedades dermatológicas usando:
- **MobileNetV2** con Transfer Learning
- **Dataset ISIC** de Kaggle (estructura de carpetas)
- **7+ clases** de condiciones cutáneas

---

## 1. Configuración del Entorno

In [ ]:
# Instalar kagglehub para descargar el dataset
!pip install kagglehub -q

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 2. Descarga del Dataset ISIC

In [ ]:
import kagglehub

# Descargar el dataset de Kaggle
path = kagglehub.dataset_download("rajivaiml/isic-skin-cancer-dataset")
print(f"📁 Dataset descargado en: {path}")

In [ ]:
# Encontrar la ruta correcta del dataset
# El dataset tiene estructura: base_path/Train/[clases] y base_path/Test/[clases]

# Buscar la carpeta que contiene Train y Test
base_path = None
for root, dirs, files in os.walk(path):
    if 'Train' in dirs and 'Test' in dirs:
        base_path = root
        break
    # También buscar variaciones comunes
    if 'train' in [d.lower() for d in dirs]:
        base_path = root
        break

if base_path is None:
    # Usar la ruta conocida del dataset ISIC
    base_path = os.path.join(path, "Skin cancer ISIC The International Skin Imaging Collaboration")

print(f"📁 Ruta base del dataset: {base_path}")
print(f"\n📂 Contenido:")
if os.path.exists(base_path):
    for item in os.listdir(base_path):
        full_path = os.path.join(base_path, item)
        if os.path.isdir(full_path):
            num_items = len(os.listdir(full_path))
            print(f"   📁 {item}/ ({num_items} elementos)")

In [ ]:
# Definir rutas de Train y Test
TRAIN_DIR = os.path.join(base_path, "Train")
TEST_DIR = os.path.join(base_path, "Test")

# Verificar que existan
print(f"✅ Train existe: {os.path.exists(TRAIN_DIR)}")
print(f"✅ Test existe: {os.path.exists(TEST_DIR)}")

# Listar clases disponibles
if os.path.exists(TRAIN_DIR):
    classes = sorted(os.listdir(TRAIN_DIR))
    print(f"\n📊 Clases encontradas ({len(classes)}):")
    for i, cls in enumerate(classes):
        cls_path = os.path.join(TRAIN_DIR, cls)
        if os.path.isdir(cls_path):
            num_images = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"   {i+1}. {cls}: {num_images} imágenes")

## 3. Configuración de Hiperparámetros

In [ ]:
# ============================================
# CONFIGURACIÓN
# ============================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001

# Número de clases (se detectará automáticamente)
NUM_CLASSES = len([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])

print(f"📊 Configuración:")
print(f"   Número de clases: {NUM_CLASSES}")
print(f"   Tamaño de imagen: {IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Épocas: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")

## 4. Crear Generadores de Datos

In [ ]:
# Data Augmentation para entrenamiento
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    fill_mode='nearest',
    validation_split=0.2  # 20% para validación
)

# Sin augmentation para validación/test
test_datagen = ImageDataGenerator(rescale=1./255)

# Generador de entrenamiento
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Generador de validación
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Generador de test
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✅ Generadores creados!")
print(f"🏋️ Imágenes de entrenamiento: {train_generator.samples}")
print(f"🧪 Imágenes de validación: {val_generator.samples}")
print(f"📝 Imágenes de test: {test_generator.samples}")
print(f"\n📊 Clases: {list(train_generator.class_indices.keys())}")

In [ ]:
# Visualizar ejemplos del dataset
plt.figure(figsize=(15, 10))
x_batch, y_batch = next(train_generator)

class_names = list(train_generator.class_indices.keys())

for i in range(min(9, len(x_batch))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_batch[i])
    class_idx = np.argmax(y_batch[i])
    plt.title(class_names[class_idx], fontsize=10)
    plt.axis('off')

plt.suptitle('Ejemplos del Dataset de Entrenamiento (con Augmentation)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Calcular Pesos de Clase (Manejo de Desbalance)

In [ ]:
# Contar imágenes por clase para calcular pesos
class_counts = {}
for class_name in train_generator.class_indices.keys():
    class_path = os.path.join(TRAIN_DIR, class_name)
    count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    class_counts[class_name] = count

print("📊 Distribución de clases en Train:")
for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cls}: {count} imágenes")

# Calcular pesos
total = sum(class_counts.values())
class_weight_dict = {}

for class_name, idx in train_generator.class_indices.items():
    # Peso inversamente proporcional a la frecuencia
    class_weight_dict[idx] = total / (len(class_counts) * class_counts[class_name])

print("\n⚖️ Pesos de clase calculados:")
for class_name, idx in train_generator.class_indices.items():
    print(f"   {class_name}: {class_weight_dict[idx]:.3f}")

## 6. Construir el Modelo CNN

In [ ]:
def build_dermaai_model(num_classes, freeze_base=True):
    """
    Construye el modelo DermaAI usando MobileNetV2 como base.
    """
    # Cargar MobileNetV2 pre-entrenado
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Congelar la base
    base_model.trainable = not freeze_base
    
    # Modelo completo
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Construir el modelo
model = build_dermaai_model(num_classes=NUM_CLASSES)
model.summary()

## 7. Entrenar el Modelo

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'dermaai_best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("🚀 Iniciando entrenamiento...\n")

In [ ]:
# Entrenar
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 8. Visualizar Resultados

In [ ]:
# Graficar curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Entrenamiento', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validación', linewidth=2)
axes[0].set_title('📈 Precisión del Modelo', fontsize=14)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Entrenamiento', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validación', linewidth=2)
axes[1].set_title('📉 Pérdida del Modelo', fontsize=14)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f"\n✅ Mejor accuracy de validación: {max(history.history['val_accuracy']):.4f}")

## 9. Evaluar en Test

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Evaluar en test
print("📊 Evaluando en conjunto de Test...")
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")
print(f"✅ Test Loss: {test_loss:.4f}")

In [ ]:
# Predicciones y matriz de confusión
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusión - Test Set', fontsize=14)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Reporte de clasificación
print("📊 Reporte de Clasificación:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## 10. Guardar el Modelo

In [ ]:
# Guardar modelo
model.save('dermaai_model_final.keras')
print("✅ Modelo guardado como 'dermaai_model_final.keras'")

model.save('dermaai_model_final.h5')
print("✅ Modelo guardado como 'dermaai_model_final.h5'")

# Guardar información de clases
import json

class_info = {
    'class_indices': train_generator.class_indices,
    'class_names': list(train_generator.class_indices.keys()),
    'num_classes': len(train_generator.class_indices)
}

with open('class_info.json', 'w') as f:
    json.dump(class_info, f, indent=2)

print("✅ Información de clases guardada en 'class_info.json'")
print(f"\n📋 Clases: {class_info['class_names']}")

## 11. Descargar Archivos

In [ ]:
# Descargar en Colab
try:
    from google.colab import files
    
    print("📥 Descargando archivos...")
    files.download('dermaai_model_final.keras')
    files.download('dermaai_model_final.h5')
    files.download('class_info.json')
    files.download('training_curves.png')
    files.download('confusion_matrix.png')
    print("✅ Descarga completada!")
except:
    print("⚠️ No estás en Colab. Archivos guardados localmente:")
    for f in ['dermaai_model_final.keras', 'dermaai_model_final.h5', 'class_info.json']:
        if os.path.exists(f):
            size = os.path.getsize(f) / (1024*1024)
            print(f"   ✓ {f} ({size:.2f} MB)")

---

## ✅ ¡Entrenamiento Completado!

### Próximos pasos:
1. Descarga los archivos a tu PC
2. Colócalos en la carpeta `models/`
3. Ejecuta el notebook `02_Train_RL_Agent.ipynb`
4. Inicia el servidor Flask: `python backend/app.py`